# Macro Indicator Ingest — pipeline parquet → quant_research (SQLite)

> **Companion spec**: `03_Macro_DB_Ingest_Design` (docs/)
> **Status**: planned skeleton — design finalised, not yet executed
> **Bridge**: input = the pipeline's `src/output/macro_panel_features.parquet` + `src/cache_vintage/*.parquet`; output = the external `quant_research` SQLite file. **Parquet remains the production SSoT; the DB is a downstream consumer.**

## Scope / prerequisites / output

**Scope**: create **`macro_panel` + `macro_panel_indicators` (macro domain)**. ⚠️ `tej_universe_returns_monthly` (equity LHS) is **out of scope for this notebook** — built by a separate equity-side aggregation step (planned) and written back to the same DB (see `03_Macro_DB_Ingest_Design` §4.3).

**Prerequisites**:
- `macro_panel_features.parquet` produced (40 indicators)
- `cache_vintage/*.parquet` produced (8 VINTAGE_SERIES first-release vintages: CPI / M2 / Core PCE + the NFCI family ×5)
- `quant_research` (SQLite) already holds the 7 `tej_*` tables (ingested by a separate TEJ flow)

**Output**:
- `macro_panel` (long-format: series × date × transform, with the `release_date` as-of-guard column)
- `macro_panel_indicators` (40 rows of static metadata)

**Idempotent**: all `INSERT OR IGNORE` — safe to re-run without duplicate writes.

In [ ]:
# ============================================================
# Cell 2 — Environment setup
# ============================================================

import os
import sqlite3
import sys
from pathlib import Path

import pandas as pd
import numpy as np
from pandas.tseries.holiday import USFederalHolidayCalendar
from pandas.tseries.offsets import CustomBusinessDay

# Pipeline root：本 notebook 預期於 <repo>/notebooks/ 下執行
PIPELINE_ROOT = Path.cwd().resolve()
if PIPELINE_ROOT.name == "notebooks":
    PIPELINE_ROOT = PIPELINE_ROOT.parent
sys.path.insert(0, str(PIPELINE_ROOT / "src"))

import config  # Indicator registry（series_id / category / lag / VINTAGE_SERIES ...）

# 外部 DB 位置：環境變數指定（各機器不同，不寫死絕對路徑）
DB_PATH = Path(os.environ.get("QUANT_RESEARCH_DB", "")).expanduser()
PARQUET_PATH = PIPELINE_ROOT / "src" / "output" / "macro_panel_features.parquet"
CACHE_VINTAGE_DIR = PIPELINE_ROOT / "src" / "cache_vintage"

assert str(DB_PATH) != ".", "請先設定環境變數 QUANT_RESEARCH_DB（指向 SQLite 檔，無 .db 副檔名）"
assert DB_PATH.exists(), f"DB not found: {DB_PATH}"
assert PARQUET_PATH.exists(), f"Parquet not found: {PARQUET_PATH}"
assert CACHE_VINTAGE_DIR.exists(), f"Cache vintage dir not found: {CACHE_VINTAGE_DIR}"

conn = sqlite3.connect(DB_PATH)
print(f"DB connected: {DB_PATH}")
print(f"Parquet path: {PARQUET_PATH}")
print(f"Config loaded: ALL_INDICATORS = {len(config.ALL_INDICATORS)} indicators")
print(f"VINTAGE_SERIES ({len(config.VINTAGE_SERIES)}): {sorted(config.VINTAGE_SERIES)}")


In [ ]:
# ============================================================
# Cell 3 — CREATE TABLE schemas
# ============================================================
# Per Macro_DB_Ingest_Design §2

schema_macro_panel = """
CREATE TABLE IF NOT EXISTS macro_panel (
    series_id     TEXT    NOT NULL,
    date          DATE    NOT NULL,
    transform     TEXT    NOT NULL,
    value         REAL,
    release_date  DATE,
    block         TEXT,
    fetched_at    TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    source        TEXT      DEFAULT 'macro_pipeline',
    PRIMARY KEY (series_id, date, transform)
)
"""

schema_indicators = """
CREATE TABLE IF NOT EXISTS macro_panel_indicators (
    series_id          TEXT    PRIMARY KEY,
    full_name          TEXT,
    fred_id            TEXT,
    block              TEXT    NOT NULL,
    block_name         TEXT,
    category           TEXT,
    publication_lag    INTEGER,
    pit_path           TEXT,
    real_lag_days      INTEGER,
    tier               INTEGER,
    is_working         INTEGER DEFAULT 1,
    fetched_at         TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    source             TEXT      DEFAULT 'macro_pipeline'
)
"""

cur = conn.cursor()
for schema in [schema_macro_panel, schema_indicators]:
    cur.execute(schema)
conn.commit()

# Scope note：本 notebook 只建 macro_panel + macro_panel_indicators（macro domain）。
# tej_universe_returns_monthly 由 factor-selection notebook step 1 建（見 Macro_DB_Ingest_Design §4.3）。

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' AND name LIKE 'macro%'", conn)
print(f"Created tables:\n{tables}")


In [ ]:
# ============================================================
# Cell 4 — Populate macro_panel_indicators (static metadata)
# ============================================================
# Per Macro_DB_Ingest_Design §2.3；欄位對齊 config.Indicator 的實際 schema
# （series_id / name / source / frequency / category / lag ...）

BLOCK_NAMES = {
    "rates": "Repo Plumbing", "liquidity": "Liquidity", "credit": "Credit",
    "curve": "Yield Curve & Real Rates", "vol": "Volatility", "leverage": "Leverage",
    "fx": "FX", "macro": "Macro Reference", "cftc": "CFTC Positioning",
}
# 真實 first-release lag（ALFRED sweep 量測值）；standard 序列以 config lag 為準
REAL_LAG_OVERRIDE = {"CPIAUCSL": 50, "M2SL": 52, "PCEPILFE": 58}

rows = []
for ind in config.ALL_INDICATORS:
    rows.append({
        "series_id": ind.series_id,
        "full_name": ind.name,
        "fred_id": ind.series_id if ind.source == "FRED" else None,
        "block": ind.category,
        "block_name": BLOCK_NAMES.get(ind.category),
        "category": ind.category,
        "publication_lag": ind.lag,
        "pit_path": "alfred_vintage" if ind.series_id in config.VINTAGE_SERIES else "standard_fixed_lag",
        "real_lag_days": REAL_LAG_OVERRIDE.get(ind.series_id, ind.lag),
        "tier": None,
        "is_working": 1,
    })

df_indicators = pd.DataFrame(rows)
df_indicators.to_sql("macro_panel_indicators", conn, if_exists="replace", index=False)
print(f"Inserted {len(df_indicators)} indicators into macro_panel_indicators")
print(df_indicators.head())


In [ ]:
# ============================================================
# Cell 5 — ingest_macro_panel(): standard fixed-lag + CFTC 規則
# ============================================================
# Per Macro_DB_Ingest_Design §3 + §4。
# 值（含 vintage 欄）一律來自 features parquet（single value source）；
# 本 cell 處理 standard / CFTC 的 release_date，vintage 欄留給 Cell 6 補首發日。

us_bday = CustomBusinessDay(calendar=USFederalHolidayCalendar())

def cftc_tff_release_date(data_date_tuesday):
    """Verified: Tuesday data → Friday 3:30 PM ET release（3 business days）；
    Friday federal holiday → release 提前至 Thursday。"""
    return pd.Timestamp(data_date_tuesday) + 3 * us_bday

def compute_release_date(series_id, data_date):
    """3 source groups 分派；VINTAGE_SERIES 由 Cell 6 處理（此處回 NaT）。"""
    if series_id in config.VINTAGE_SERIES:
        return pd.NaT
    if series_id.startswith("TFF_"):
        return cftc_tff_release_date(data_date)
    ind = config.get(series_id)  # config 的「找唯一」helper
    if ind is None or ind.lag is None:
        return pd.NaT
    return pd.Timestamp(data_date) + pd.Timedelta(days=ind.lag)

def _block(series_id):
    ind = config.get(series_id)
    return ind.category if ind is not None else "unknown"

def parse_series_transform(col_name):
    """'SOFR' → ('SOFR','level')；'SOFR_pct1m' → ('SOFR','pct1m')；
    'BAMLH0A0HYM2_diff_bps' → ('BAMLH0A0HYM2','diff_bps')。長字尾優先比對。"""
    KNOWN_TRANSFORMS = ["diff_bps", "pct1m", "yoy", "zscore", "diff"]
    for tx in KNOWN_TRANSFORMS:
        if col_name.endswith(f"_{tx}"):
            return pd.Series([col_name[: -len(tx) - 1], tx])
    return pd.Series([col_name, "level"])

# Read parquet（wide）→ melt（long）
df_wide = pd.read_parquet(PARQUET_PATH)
df_wide = df_wide.reset_index().rename(columns={df_wide.index.name or "index": "date"})
df_long_all = df_wide.melt(id_vars=["date"], var_name="series_transform", value_name="value")
df_long_all[["series_id", "transform"]] = df_long_all["series_transform"].apply(parse_series_transform)

# 分流：standard/CFTC 本 cell；vintage 留給 Cell 6
df_long = df_long_all[~df_long_all["series_id"].isin(config.VINTAGE_SERIES)].copy()

df_long["release_date"] = df_long.apply(
    lambda row: compute_release_date(row["series_id"], row["date"]), axis=1
)
df_long["block"] = df_long["series_id"].apply(_block)

df_long_clean = df_long.dropna(subset=["value"]).copy()
df_long_clean["date"] = pd.to_datetime(df_long_clean["date"]).dt.strftime("%Y-%m-%d")
df_long_clean["release_date"] = pd.to_datetime(df_long_clean["release_date"]).dt.strftime("%Y-%m-%d")

cur = conn.cursor()
rows_to_insert = df_long_clean[["series_id", "date", "transform", "value", "release_date", "block"]].values.tolist()
cur.executemany(
    """INSERT OR IGNORE INTO macro_panel
       (series_id, date, transform, value, release_date, block)
       VALUES (?, ?, ?, ?, ?, ?)""",
    rows_to_insert,
)
conn.commit()
print(f"Inserted {cur.rowcount} rows into macro_panel (standard fixed-lag + CFTC paths)")


In [ ]:
# ============================================================
# Cell 6 — VINTAGE_SERIES：release_date = as-of 首發日（值仍取自 features parquet）
# ============================================================
# Per Macro_DB_Ingest_Design §5：本 cell **不重算任何 transform**（single value source）。
# vintage 欄的值（level + 各 transform）已由 Cell 5 的 melt 產生；此處只補正確的 release_date —
# 對每個 grid date，release_date = 「realtime_start ≤ 該日」的最近一次首發（merge_asof backward）。

# 1) 首發日 lookup（per series）：cache_vintage long-format → first release per data_date
first_release = {}
for sid in sorted(config.VINTAGE_SERIES):
    p = CACHE_VINTAGE_DIR / f"{sid}.parquet"
    if not p.exists():
        print(f"WARN: {p} not found, skipping {sid}")
        continue
    dfv = pd.read_parquet(p)
    fr = (dfv.sort_values(["date", "realtime_start"], kind="stable")
             .drop_duplicates(subset=["date"], keep="first"))
    first_release[sid] = fr[["realtime_start"]].sort_values("realtime_start").reset_index(drop=True)

# 2) 取回 Cell 5 分流出的 vintage rows，as-of 併上首發日
parts = []
for sid, look in first_release.items():
    sub = df_long_all[df_long_all["series_id"] == sid].dropna(subset=["value"]).copy()
    if sub.empty:
        continue
    sub["date"] = pd.to_datetime(sub["date"])
    merged = pd.merge_asof(
        sub.sort_values("date"), look,
        left_on="date", right_on="realtime_start", direction="backward",
    )
    merged["release_date"] = merged["realtime_start"]
    parts.append(merged)

df_v = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
if not df_v.empty:
    df_v = df_v.dropna(subset=["release_date"])  # 首發前 warm-up 期不落 DB
    df_v["block"] = df_v["series_id"].apply(_block)
    df_v["date"] = df_v["date"].dt.strftime("%Y-%m-%d")
    df_v["release_date"] = pd.to_datetime(df_v["release_date"]).dt.strftime("%Y-%m-%d")
    cur = conn.cursor()
    cur.executemany(
        """INSERT OR IGNORE INTO macro_panel
           (series_id, date, transform, value, release_date, block)
           VALUES (?, ?, ?, ?, ?, ?)""",
        df_v[["series_id", "date", "transform", "value", "release_date", "block"]].values.tolist(),
    )
    conn.commit()
    print(f"VINTAGE_SERIES ingest complete: {len(df_v)} rows（值來自 features parquet；release_date = as-of 首發日）")
else:
    print("VINTAGE_SERIES ingest: no rows（檢查 cache_vintage/ 是否已產出）")


In [ ]:
# ============================================================
# Cell 7 — Verify SELECT queries + monitoring SQL
# ============================================================
# Per Macro_DB_Ingest_Design §7

# Q1: 每個 transform 多少 rows?
print("\n=== Q1: rows by transform ===")
print(pd.read_sql("SELECT transform, COUNT(*) AS n FROM macro_panel GROUP BY transform", conn))

# Q2: 每個 block 幾個 series + total rows?
print("\n=== Q2: by block ===")
print(pd.read_sql("""
SELECT block, COUNT(DISTINCT series_id) AS n_series, COUNT(*) AS n_rows
FROM macro_panel
GROUP BY block
ORDER BY block
""", conn))

# Q3: release_date vs date gap per series
print("\n=== Q3: lag distribution per series (level transform) ===")
print(pd.read_sql("""
SELECT
    series_id,
    ROUND(AVG(julianday(release_date) - julianday(date)), 1) AS avg_lag_days,
    MAX(julianday(release_date) - julianday(date)) AS max_lag_days,
    MIN(julianday(release_date) - julianday(date)) AS min_lag_days
FROM macro_panel
WHERE transform = 'level'
GROUP BY series_id
ORDER BY avg_lag_days DESC
LIMIT 20
""", conn))

# Q4: ALFRED vintage vs standard fixed-lag 對照
print("\n=== Q4: ALFRED vintage vs standard fixed-lag (VINTAGE_SERIES check) ===")
print(pd.read_sql("""
SELECT
    m.series_id,
    i.pit_path,
    i.publication_lag AS std_config_lag,
    ROUND(AVG(julianday(m.release_date) - julianday(m.date)), 1) AS empirical_lag,
    i.real_lag_days AS sweep_lag
FROM macro_panel m
JOIN macro_panel_indicators i ON m.series_id = i.series_id
WHERE m.transform = 'level' AND m.series_id IN ('CPIAUCSL', 'M2SL', 'PCEPILFE')
GROUP BY m.series_id, i.pit_path, i.publication_lag, i.real_lag_days
""", conn))

# Q5: as-of JOIN preview（RHS macro × LHS equity，無 look-ahead）
print("\n=== Q5: as-of JOIN preview (HY OAS zscore × equal_weight LHS) ===")
print(pd.read_sql("""
SELECT
    u.year_month,
    u.universe_label,
    u.monthly_return AS lhs_return,
    m.value AS rhs_hy_oas_zscore,
    m.release_date AS rhs_release_date
FROM tej_universe_returns_monthly u
LEFT JOIN macro_panel m
    ON m.series_id = 'BAMLH0A0HYM2'
    AND m.transform = 'zscore'
    AND m.release_date <= date(u.year_month || '-01')
WHERE u.universe_label = 'equal_weight'
ORDER BY u.year_month DESC
LIMIT 10
""", conn))

conn.close()
print("\n=== Macro ingest complete ===")


## Post-execution checklist

- [ ] Cells 1–7 run clean, no errors
- [ ] Q1 verify: transform set ⊆ {level, diff, diff_bps, pct1m, yoy}
- [ ] Q2 verify: all 9 blocks have series (rates / liquidity / credit / curve / vol / leverage / fx / macro / cftc)
- [ ] Q3 verify: daily / weekly series avg_lag ≈ 0–5 days (calibrated per the ALFRED sweep)
- [ ] Q4 verify: CPI / M2 / PCEPILFE empirical_lag ≥ the measured first-release lag (≈ 42 / 55 / 58; grid ffill reads slightly above the first-release value)
- [ ] Q5 verify: `rhs_release_date <= year_month-01`, no look-ahead
- [ ] Open `quant_research` in a SQL client and confirm `macro_panel` + `macro_panel_indicators` exist
  (`tej_universe_returns_monthly` is built by a separate equity-side step)

## Downstream

A separate equity-side step builds `tej_universe_returns_monthly`, which then joins `macro_panel` in an as-of SQL JOIN
(see `03_Macro_DB_Ingest_Design` §4.3); Fama-MacBeth pulls LHS + RHS from the same DB.

---

*Last updated: 2026-08-04*